In [1]:
import numpy as np
import pandas as pd 
import cmath 
import glob
import scipy.special as sp
import scipy.signal as spg
import scipy.constants as const
import matplotlib.pyplot as plt
plt.style.use('dark_background')
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['axes.titlesize'] = 24
plt.rcParams['legend.fontsize'] = 8
import VNA_data_read_local as vdr
import useful_functions as uf

In [2]:
# regular data 
baselines = glob.glob('/Users/leayamashiro/whispering_gallery_MAIN/calibrated/data/data_060925/baselines/*.txt')
rot1 = glob.glob('/Users/leayamashiro/whispering_gallery_MAIN/calibrated/data/data_060925/rot1/*.txt')
rot1FL = glob.glob('/Users/leayamashiro/whispering_gallery_MAIN/calibrated/data/data_060925/rot1FL/*.txt')
rot3 = glob.glob('/Users/leayamashiro/whispering_gallery_MAIN/calibrated/data/data_060925/rot3/*.txt')
rot5 = glob.glob('/Users/leayamashiro/whispering_gallery_MAIN/calibrated/data/data_060925/rot5/*.txt')

# vertical disk data
vert_baselines = glob.glob('/Users/leayamashiro/whispering_gallery_MAIN/calibrated/data/data_060925/vert_baselines/*.txt')
vert = glob.glob('/Users/leayamashiro/whispering_gallery_MAIN/calibrated/data/data_060925/vert/*.txt')


In [6]:
baseline_2_1 = 'baselines/LSB_BL2_1_2.5to6.18GHz_2025-06-09_14-17-30.txt'
data_folder_path = 'calibrated/data/data_060925'

In [35]:

def get_dips_data(BL, disk, n_dips, f_start=None, f_stop=None, title = 'title'): # for S21 dips in VNA data, need to already have baseline & disk data loaded in as variables

    # prepare signal data 
    S21_subtracted = (20*np.log10(np.abs(disk['Complex (decimal)']))
                  -20*np.log10(np.abs(BL['Complex (decimal)']))) # just to get the calibrated one ready
    S21_freqs = 1e-9*BL['Freq (Hz)'] # convert to GHz
    S21_sub = pd.DataFrame({'freqs':S21_freqs, 'S21':S21_subtracted}) # make calibrated data dictionary
    if (f_start is not None) and (f_stop is not None): 
        S21_subt = S21_sub[(S21_sub['freqs']>=f_start) & (S21_sub['freqs']<=f_stop)]
    else: 
        S21_subt = S21_sub
    # peak finding
    S21_dips, _dips = spg.find_peaks(-S21_subt['S21']) # negative because need to flip
    dip_freqs = S21_subt['freqs'].iloc[S21_dips] # get frequency values for dips
    dip_S21 = S21_subt['S21'].iloc[S21_dips] # get S21 of the located dips
    dip_dict = {'freqs': dip_freqs, 'dip S21': dip_S21} # make dip dictionary
    dips_sorted = pd.DataFrame(dip_dict).sort_values('dip S21', ascending=True).reset_index(inplace=False) # make DF where dips sorted by mag
    top_dips = dips_sorted.iloc[0:n_dips] # grab top 10 deepest dips

    return S21_subt, dips_sorted # returns the baseline-subtracted signal data and the dips in a sorted table



# function to streamline this
def folder_loader_dips_data(baseline_file, glob_loaded, data_folder_path, f_start=None, f_stop=None):
    main_directory_path = '/Users/leayamashiro/whispering_gallery_MAIN/'

    run_dict = {}

    for i in range(len(glob_loaded)): 
        # for ID-ing run parameters and printing in plots
        test_run = glob_loaded[i].split('/')[-1]
        run_name_split = test_run.split('_')
        run_name_for_plot = run_name_split[0] + '_' + run_name_split[1] + '_' + run_name_split[2]
        # loading in as data 
        baseline = uf.just_single_loader(main_directory_path + data_folder_path + '/' + baseline_file)
        disk = uf.just_single_loader(main_directory_path + data_folder_path + '/' + run_name_split[1] + '/' + test_run)
        # plotting 
        signal_data, dips_table = get_dips_data(BL = baseline, 
                                                disk = disk,
                                                n_dips = 10, 
                                                f_start=f_start, 
                                                f_stop=f_stop,
                                                title = run_name_for_plot)
        
        run_dict[i] = {'run_name': run_name_for_plot, 
                       'signal data': signal_data, 
                       'dips table': dips_table}
        
    return run_dict # remember it's 1. name of the run, 2. the signal data, and 3. the dips table

In [29]:
rot1_data = folder_loader_dips_data(baseline_2_1, rot1, data_folder_path)
rot1FL_data = folder_loader_dips_data(baseline_2_1, rot1FL, data_folder_path)
rot3_data = folder_loader_dips_data(baseline_2_1, rot3, data_folder_path)
rot5_data = folder_loader_dips_data(baseline_2_1, rot5, data_folder_path)


In [43]:
rot1_data[1]

{'run_name': 'LSB_rot1_Submm2',
 'signal data':        freqs       S21
 0     2.5000 -0.755083
 1     2.5023 -0.803344
 2     2.5046 -0.755747
 3     2.5069 -0.422051
 4     2.5092 -0.262305
 ...      ...       ...
 1596  6.1708 -0.099473
 1597  6.1731 -0.096617
 1598  6.1754 -0.211055
 1599  6.1777 -0.257769
 1600  6.1800 -0.199214
 
 [1601 rows x 2 columns],
 'dips table':      index   freqs   dip S21
 0     1566  6.1018 -4.071788
 1     1486  5.9178 -3.637463
 2     1399  5.7177 -3.114886
 3     1363  5.6349 -3.070069
 4     1269  5.4187 -3.028407
 ..     ...     ...       ...
 390    990  4.7770  1.073576
 391   1010  4.8230  1.073787
 392   1055  4.9265  1.083642
 393    994  4.7862  1.107407
 394    997  4.7931  1.133873
 
 [395 rows x 3 columns]}

In [ ]:
rot1_1mm = rot1_data[0]
rot1_submm2 = rot1_data[1]

{'run_name': 'LSB_rot1_1mm',
 'signal data':        freqs       S21
 0     2.5000 -0.226873
 1     2.5023 -0.196298
 2     2.5046 -0.319487
 3     2.5069 -0.260455
 4     2.5092 -0.350840
 ...      ...       ...
 1596  6.1708  0.550107
 1597  6.1731  0.386012
 1598  6.1754  0.348984
 1599  6.1777  0.307533
 1600  6.1800  0.352376
 
 [1601 rows x 2 columns],
 'dips table':      index   freqs   dip S21
 0     1439  5.8097 -4.458681
 1     1576  6.1248 -3.432435
 2     1567  6.1041 -2.272236
 3     1093  5.0139 -2.033038
 4     1487  5.9201 -1.806639
 ..     ...     ...       ...
 418    632  3.9536  0.345809
 419   1066  4.9518  0.350338
 420   1117  5.0691  0.368416
 421   1595  6.1685  0.391168
 422   1113  5.0599  0.398347
 
 [423 rows x 3 columns]}

In [36]:
rot1_dips = rot1_data[0]['dips table']

KeyError: 'signal data'

In [ ]:
# combining two functions for more ease looking at data tables

def folder_loader_dips_data_runs(baseline_file, glob_loaded, data_folder_path, f_start=None, f_stop=None):
    main_directory_path = '/Users/leayamashiro/whispering_gallery_MAIN/'

    signal_runs_dict = []
    dips_table_runs_dict = []

    for i in range(len(glob_loaded)): 
        # for ID-ing run parameters and printing in plots
        test_run = glob_loaded[i].split('/')[-1]
        run_name_split = test_run.split('_')
        run_name_for_plot = run_name_split[0] + '_' + run_name_split[1] + '_' + run_name_split[2]
        # loading in as data 
        baseline = uf.just_single_loader(main_directory_path + data_folder_path + '/' + baseline_file)
        disk = uf.just_single_loader(main_directory_path + data_folder_path + '/' + run_name_split[1] + '/' + test_run)
        
        # set the data pulling and find peaks 
        def get_dips_data_runs(BL, disk, n_dips, f_start=None, f_stop=None, title = 'title'): # for S21 dips in VNA data, need to already have baseline & disk data loaded in as variables

            # prepare signal data 
            S21_subtracted = (20*np.log10(np.abs(disk['Complex (decimal)']))
                        -20*np.log10(np.abs(BL['Complex (decimal)']))) # just to get the calibrated one ready
            S21_freqs = 1e-9*BL['Freq (Hz)'] # convert to GHz
            S21_sub = pd.DataFrame({'freqs':S21_freqs, 'S21':S21_subtracted}) # make calibrated data dictionary
            if (f_start is not None) and (f_stop is not None): 
                S21_subt = S21_sub[(S21_sub['freqs']>=f_start) & (S21_sub['freqs']<=f_stop)]
            else: 
                S21_subt = S21_sub
            # peak finding
            S21_dips, _dips = spg.find_peaks(-S21_subt['S21']) # negative because need to flip
            dip_freqs = S21_subt['freqs'].iloc[S21_dips] # get frequency values for dips
            dip_S21 = S21_subt['S21'].iloc[S21_dips] # get S21 of the located dips
            dip_dict = {'freqs': dip_freqs, 'dip S21': dip_S21} # make dip dictionary
            dips_sorted = pd.DataFrame(dip_dict).sort_values('dip S21', ascending=True).reset_index(inplace=False) # make DF where dips sorted by mag
            signal_with_run_name = S21_subt.copy()
            signal_with_run_name['run name'] = run_name_for_plot
            dips_with_run_name = dips_sorted.copy()
            dips_with_run_name['run name'] = run_name_for_plot
            top_dips = dips_sorted.iloc[0:n_dips] # grab top 10 deepest dips

            return signal_with_run_name, dips_with_run_name # returns the baseline-subtracted signal data and the dips in a sorted table

        signal_data, dips_table = get_dips_data_runs(BL = baseline, 
                                                disk = disk,
                                                n_dips = 10, 
                                                f_start=f_start, 
                                                f_stop=f_stop,
                                                title = run_name_for_plot)
        
        signal_runs_dict.append(signal_data)
        dips_table_runs_dict.append(dips_table)
        
    return signal_runs_dict, dips_table_runs_dict

In [64]:
rot1_signals, rot1_dips_tables = folder_loader_dips_data_runs(baseline_2_1, rot1, data_folder_path, f_start=3, f_stop=4)

In [56]:
rot1_signals[0]

,freqs,S21,run name
0,2.5000,-0.226873,LSB_rot1_1mm
1,2.5023,-0.196298,LSB_rot1_1mm
2,2.5046,-0.319487,LSB_rot1_1mm
3,2.5069,-0.260455,LSB_rot1_1mm
4,2.5092,-0.350840,LSB_rot1_1mm
...,...,...,...
1596,6.1708,0.550107,LSB_rot1_1mm
1597,6.1731,0.386012,LSB_rot1_1mm
1598,6.1754,0.348984,LSB_rot1_1mm
1599,6.1777,0.307533,LSB_rot1_1mm


In [66]:
for i in range(len(rot1_dips_tables)):
    print(rot1_dips_tables[i].head(20))

    index   freqs   dip S21      run name
0     327  3.2521 -0.958103  LSB_rot1_1mm
1     245  3.0635 -0.666838  LSB_rot1_1mm
2     405  3.4315 -0.632397  LSB_rot1_1mm
3     381  3.3763 -0.590747  LSB_rot1_1mm
4     403  3.4269 -0.570300  LSB_rot1_1mm
5     401  3.4223 -0.568721  LSB_rot1_1mm
6     315  3.2245 -0.561954  LSB_rot1_1mm
7     355  3.3165 -0.548147  LSB_rot1_1mm
8     359  3.3257 -0.541800  LSB_rot1_1mm
9     357  3.3211 -0.527945  LSB_rot1_1mm
10    361  3.3303 -0.515459  LSB_rot1_1mm
11    410  3.4430 -0.515270  LSB_rot1_1mm
12    312  3.2176 -0.513952  LSB_rot1_1mm
13    335  3.2705 -0.492695  LSB_rot1_1mm
14    490  3.6270 -0.475079  LSB_rot1_1mm
15    318  3.2314 -0.474764  LSB_rot1_1mm
16    337  3.2751 -0.469641  LSB_rot1_1mm
17    376  3.3648 -0.461079  LSB_rot1_1mm
18    320  3.2360 -0.460711  LSB_rot1_1mm
19    391  3.3993 -0.431339  LSB_rot1_1mm
    index   freqs   dip S21         run name
0     326  3.2498 -2.592066  LSB_rot1_Submm2
1     310  3.2130 -2.167899 